# 02. 动手：自建 Skill 系统

把前两个 notebook 合起来：一个能扫描、选择、注入 skill 的最小系统，外加 harness 的 hook 保护。
这个 notebook 的终点是一个可复用的 `MiniSkillAgent` 类。

> **检查点**：能说出这个 mini 系统与生产 harness（Claude Code）相比缺什么（持久化、权限文件、MCP 接入、真正的进程隔离）。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# MiniSkillAgent：skill 加载 + hook 保护 + LLM 驱动的最小闭环
import re
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
SKILLS_DIR = ROOT / 'agents' / 'agent-skills' / 'skills'

def parse_frontmatter(text):
    m = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)$", text, re.DOTALL)
    meta = dict(line.split(': ', 1) for line in m.group(1).splitlines() if ': ' in line)
    return meta, m.group(2).strip()

class MiniSkillAgent:
    def __init__(self, client, skills_dir):
        self.client = client
        self.skills = []
        for p in sorted(skills_dir.glob('*/SKILL.md')):
            meta, body = parse_frontmatter(p.read_text())
            self.skills.append({'name': meta.get('name'), 'description': meta.get('description', ''), 'body': body})
        # Layer 1: 常驻 system prompt 的只有 metadata
        catalog = "\n".join(f"- {s['name']}: {s['description']}" for s in self.skills)
        self.base_system = f"你是一个助手。你可以使用以下 skills（需要时声明）：\n{catalog}"

    def run(self, query: str) -> str:
        if self.client is None:
            return f"[toy] 收到 '{query}'，已加载 {len(self.skills)} 个 skill 的 metadata"
        # 第一步：LLM 判断是否需要 skill（只看到 metadata）
        pick = chat(self.client, f"{query}\n\n如果需要某个 skill，只回答 skill 名；否则回答 NONE。",
                    system=self.base_system, max_tokens=50)
        name = pick.strip().strip('"`').split()[0] if pick.strip() else "NONE"
        skill = next((s for s in self.skills if s['name'] == name), None)
        if not skill:
            print(f"未触发 skill，直接回答")
            return chat(self.client, query, system="You are a helpful assistant.")
        # 第二步：触发后注入 body（Layer 2）
        print(f"触发 skill: {skill['name']}（注入 {len(skill['body'])//4} tokens）")
        return chat(self.client, query, system=f"{self.base_system}\n\n已激活 skill，严格遵守：\n{skill['body']}")

agent = MiniSkillAgent(client, SKILLS_DIR)
print(f"已扫描 {len(agent.skills)} 个 skill")
print(agent.run("1 GB 等于多少 MiB？注意我说的是十进制还是二进制？"))

已扫描 1 个 skill


未触发 skill，直接回答


1 GB（十进制，即 10^9 字节）等于约 **953.674 MiB**（二进制，即 2^20 字节）。因为 1 GB = 1,000,000,000 字节，而 1 MiB = 1,048,576 字节，所以换算结果为 1,000,000,000 ÷ 1,048,576 ≈ 953.674。你提到的 GB 是十进制单位，MiB 是二进制单位。


## 与生产 Harness 的差距

| 能力 | MiniSkillAgent | Claude Code 级 harness |
|------|---------------|----------------------|
| Skill 选择 | 一次 LLM 调用 | 描述匹配 + 模型自主 + 用户显式 / |
| 资源层 | 无 | 正文引用文件按需读 |
| Hook | 无（上一个 notebook 有 demo） | Pre/PostToolUse 全生命周期 |
| 持久化 | 无 | 会话恢复、上下文压缩 |
| 权限 | 无 | allow/deny/ask + 配置文件 |

In [3]:
# 触发率自测：5 个应触发 + 2 个不应触发
should_trigger = ["3.5 磅是多少公斤", "convert 10 miles to km", "100°F 是几度", "1 GiB 是多少 MiB", "把 72 华氏度换成摄氏度"]
should_not = ["写一首关于秋天的诗", "Python 怎么读 csv"]

for q in should_trigger:
    pick = chat(client, f"{q}\n\n如果需要 unit-converter skill 只回答它的名字，否则回答 NONE。",
                system=agent.base_system, max_tokens=30) if client else "unit-converter"
    hit = "unit-converter" in pick
    print(f"{'✅' if hit else '❌'} 应触发: {q} → {pick.strip()[:40]}")

for q in should_not:
    pick = chat(client, f"{q}\n\n如果需要 unit-converter skill 只回答它的名字，否则回答 NONE。",
                system=agent.base_system, max_tokens=30) if client else "NONE"
    ok = "unit-converter" not in pick
    print(f"{'✅' if ok else '❌'} 不应触发: {q} → {pick.strip()[:40]}")

❌ 应触发: 3.5 磅是多少公斤 → 


❌ 应触发: convert 10 miles to km → 


❌ 应触发: 100°F 是几度 → 


❌ 应触发: 1 GiB 是多少 MiB → 


✅ 应触发: 把 72 华氏度换成摄氏度 → unit-converter


✅ 不应触发: 写一首关于秋天的诗 → N


✅ 不应触发: Python 怎么读 csv → 


## 练习

1. 给 `MiniSkillAgent` 加 Layer 3：skill 正文里写「详细系数表见 resources/ratios.md」，让 agent 在需要时读取该文件。
2. 写第二个 skill（如 `code-namer`：为变量/函数起名），测试两个 skill 并存时的选择准确率。
3. 把触发测试扩到 20 个问题，计算 precision/recall；针对误触发优化 description，再测一轮 — 这就是 skill 的「评估驱动开发」。